# Lab 01: Text Steganography (Part 1 - Inter-Word Spacing in Justified Text)

**Course:** Information Hiding & Secret Sharing

**Topic:** White Space Methods - Justified Text Steganography

---

**Student Name:** Lê Thị Hồng Hạnh

**Student ID:** 22127103

---
## 1. Introduction

### 1.1 Background

In justified text, extra spaces are added between words to align both left and right edges evenly. This creates a natural opportunity to hide secret data by controlling **where** the extra spaces are placed.

### 1.2 Encoding Scheme

We use **pair-based encoding** to embed bits:

| Bit | Pattern | Description |
|-----|---------|-------------|
| **0** | (2 spaces, 1 space) | Extra space goes to FIRST gap in pair |
| **1** | (1 space, 2 spaces) | Extra space goes to SECOND gap in pair |
| **No embed** | Other patterns | Line not used or capacity exhausted |

### 1.3 Key Variables

For each line:
- **n** = Number of inter-word gaps (= number of words - 1)
- **m** = Number of extra spaces needed to justify the line = `text_width - len(line with single spaces)`
- **Capacity** = `min(m, n - m)` bits (when `0 < m < n`, otherwise 0)

### 1.4 Termination Pattern

After embedding all message bits, append:
- One `1` bit (termination marker)
- Fill remaining capacity with `0` bits

This creates tail pattern: `100...`

When extracting, find the last `1` and remove everything from there to the end.

---
## 2. Instructions

### How to Complete This Lab

⚠️ **Important:** This notebook will be graded using automated testing. Please follow the instructions carefully.

**Completing the Tasks**

1. Fill in your name and student ID at the top of this notebook.

2. Complete the code in cells marked with:
```python
# YOUR CODE HERE
raise NotImplementedError()
```

3. Delete the `raise NotImplementedError()` line when you implement your solution.

4. For optional tasks:
```python
# YOUR CODE HERE (OPTIONAL)
```

5. For written answers:
```markdown
YOUR ANSWER HERE
```

**Testing Your Code**

- Below each task, there are test cells to verify your implementation.
- If a test cell runs without errors, your code passes that test.
- ⚠️ Passing all visible tests does not guarantee full correctness - there may be hidden test cases.

**Before Submission**

1. Run `Kernel` → `Restart Kernel & Run All Cells` to ensure everything works.
2. Remove any debug print statements or extra cells you created.
3. **Do not modify** the provided code cells or test cells.

**Submission Format**

```
StudentID_Lab1/
└── Part1/
    └── Lab01_Text_Steganography_Part1.ipynb
└── Part2/

```

Compress the `StudentID` folder and submit.

---

**Academic Integrity**

🎯 The goal is to **learn authentically**. You may discuss ideas with classmates, but your code and answers must be your own work based on your genuine understanding.

🚫 **Plagiarism or cheating will result in a score of 0 for the entire course.**

---
## 3. Helper Functions

The following helper functions are provided for converting between text and bits.

In [1]:
def text_to_bits(text: str) -> str:
    """
    Convert a text string to a binary string using UTF-8 encoding.
    
    Args:
        text: The text string to convert
    
    Returns:
        Binary string (e.g., "Hi" → "0100100001101001")
    
    Example:
        >>> text_to_bits('A')
        '01000001'
        >>> text_to_bits('Hi')
        '0100100001101001'
    """
    if not text:
        return ''
    
    # Encode text to UTF-8 bytes
    text_bytes = text.encode('utf-8')
    
    # Convert each byte to 8-bit binary string
    bits = ''.join(format(byte, '08b') for byte in text_bytes)
    
    return bits


def bits_to_text(bits: str) -> str:
    """
    Convert a binary string back to text using UTF-8 encoding.
    
    Args:
        bits: Binary string (must be valid UTF-8 when decoded)
    
    Returns:
        Decoded text string
    
    Example:
        >>> bits_to_text('01000001')
        'A'
        >>> bits_to_text('0100100001101001')
        'Hi'
    """
    bits = bits[:len(bits) // 8 * 8]
    if not bits:
        return ''
    try:        
        return bytes(int(bits[i:i+8], 2) for i in range(0, len(bits), 8)).decode('utf-8')
    except:
        return ''

In [2]:
# Test helper functions
assert text_to_bits('A') == '01000001'
assert bits_to_text('01000001') == 'A'
assert bits_to_text(text_to_bits('Hello')) == 'Hello'
print("Helper functions are working correctly!")

Helper functions are working correctly!


---
## 4. Task 1: Parse Line (0.25 pts)

Implement a function to parse a justified line into words and gap sizes.

**Example:**
```
Input:  "The  quick fox"
Output: (['The', 'quick', 'fox'], [2, 1])
```

In [3]:
def parse_line(line):
    """
    Parse a line into words and gap sizes.
    
    Args:
        line (str): A string with words separated by one or more spaces.
    
    Returns:
        tuple: (list of words, list of gap sizes)
               gap sizes[i] = number of spaces between words[i] and words[i+1]
    
    Examples:
        >>> parse_line("The  quick fox")
        (['The', 'quick', 'fox'], [2, 1])
        >>> parse_line("Hello   world")
        (['Hello', 'world'], [3])
    """
    # YOUR CODE HERE
    words = []
    gaps = []
    
    # Current word and gap in loop
    current_word = ''
    current_gap = 0
    
    # Loop through the line
    for c in line:
        # The current character is not a space
        if c != ' ':
            # Start of new word after a gap
            if current_gap and not current_word:
                gaps.append(current_gap)
                current_gap = 0
            
            current_word += c
        # The current character is a space
        else:
            # Start of new gap after a word
            if current_word and not current_gap:
                words.append(current_word)
                current_word = ''
                
            current_gap += 1
            
    if current_word:
        words.append(current_word)

    return words, gaps

In [4]:
# TEST: Parse Line
words, gaps = parse_line("The  quick fox")
assert words == ['The', 'quick', 'fox'], f"Words incorrect: {words}"
assert gaps == [2, 1], f"Gaps incorrect: {gaps}"

words, gaps = parse_line("Hello   world")
assert words == ['Hello', 'world'], f"Words incorrect: {words}"
assert gaps == [3], f"Gaps incorrect: {gaps}"

words, gaps = parse_line("Single")
assert words == ['Single'], f"Words incorrect: {words}"
assert gaps == [], f"Gaps incorrect: {gaps}"

words, gaps = parse_line("A  B  C  D")
assert words == ['A', 'B', 'C', 'D'], f"Words incorrect: {words}"
assert gaps == [2, 2, 2], f"Gaps incorrect: {gaps}"

print("All parse_line tests passed! ✓")

All parse_line tests passed! ✓


---
## 5. Task 2: Create Line (0.25 pts)

Implement a function to create a justified line from words and gap sizes.

**Example:**
```
Input:  words=['The', 'quick', 'fox'], gap_sizes=[2, 1]
Output: "The  quick fox"
```

In [5]:
def create_line(words, gap_sizes):
    """
    Create a line from words with specified gap sizes.
    
    Args:
        words (list): List of words.
        gap_sizes (list): List of space counts between words.
                         Length should be len(words) - 1.
    
    Returns:
        str: The constructed line with words separated by specified spaces.
    
    Examples:
        >>> create_line(['The', 'quick', 'fox'], [2, 1])
        'The  quick fox'
        >>> create_line(['Hello', 'world'], [3])
        'Hello   world'
    """
    # YOUR CODE HERE
    if not words:
        return ''
    
    line = words[0]
    for i, gap in enumerate(gap_sizes):
        line += ' ' * gap + words[i + 1]
    return line

In [6]:
# TEST: Create Line
assert create_line(['The', 'quick', 'fox'], [2, 1]) == 'The  quick fox'
assert create_line(['Hello', 'world'], [3]) == 'Hello   world'
assert create_line(['Single'], []) == 'Single'
assert create_line(['A', 'B', 'C'], [1, 1]) == 'A B C'
assert create_line(['A', 'B', 'C'], [2, 2]) == 'A  B  C'

# Test round-trip: parse then create should give original
original = "The  quick  brown fox"
words, gaps = parse_line(original)
reconstructed = create_line(words, gaps)
assert reconstructed == original, f"Round-trip failed: {reconstructed} != {original}"

print("All create_line tests passed! ✓")

All create_line tests passed! ✓


---
## 6. Task 3: Embed Function (2 points)

Implement the main embedding function that hides a secret message in cover text.

**Requirements:**
1. Read the secret message from `msg_file`
2. Read the cover text from `cover_text_file`
3. Convert the message to bits and add termination pattern (`1` followed by `0`s)
4. Embed bits using pair-based encoding in justified lines
5. Write the stego text to `stego_text_file`
6. Return `True` if successful, `False` if not enough capacity

**Encoding Rules:**
- For each pair of consecutive gaps (gap_i, gap_{i+1}):
  - Bit `0`: Add extra space to first gap → pattern (2, 1)
  - Bit `1`: Add extra space to second gap → pattern (1, 2)
- Distribute remaining extra spaces evenly to remaining gaps

In [7]:
## This function is becoming quite long; 
## please feel free to decompose it into smaller helper functions as you see fit.
def is_aligned_line(l, lines):
    # Index out of range or index of the last line
    if l >= len(lines) - 1:
        return False
    
    # Not enough the number of word in a line to embed
    if len(lines[l].split()) < 2:
        return False
    
    # The end line of paragraph
    if lines[l + 1].strip() == '':
        return False

    return True

def embed(msg_file, cover_text_file, text_width, stego_text_file):
    """
    Embed a secret message into cover text using inter-word spacing.
    
    Args:
        msg_file (str): Path to file containing the secret message.
        cover_text_file (str): Path to file containing the cover text.
        text_width (int): Target line width for justified text.
        stego_text_file (str): Path to output file for stego text.
    
    Returns:
        bool: True if embedding succeeded, False if not enough capacity.
    
    Notes:
        - Preserves original line structure (paragraphs, empty lines).
        - Lines at end of paragraph (before empty line) are NOT justified.
        - Only mid-paragraph lines are justified to text_width.
    """
    # YOUR CODE HERE
    # Read secret message and cover text
    with open(msg_file, 'r') as f:
        msg_text = f.read()
    with open(cover_text_file, 'r') as f:
        cover_text = f.read()
    
    # Convert message to bit string and cover text to array of lines
    msg_bits = text_to_bits(msg_text)
    cover_lines = cover_text.splitlines()
    
    # Init indexes in loop
    b = 0 # Index of msg_bits
    l = 0 # Index of covers_lines
    
    # Check if bit 1 has been added to the tail
    marker_bit_embedded = False
    
    stego_text = ''
    while l < len(cover_lines):
        stego_line = cover_lines[l]
        if is_aligned_line(l, cover_lines):
            words, gaps = parse_line(cover_lines[l])
            n = len(gaps)
            m = text_width - len(cover_lines[l])
            
            if 0 < m < n:
                capacity = min(m, n - m)
                for i in range(capacity):
                    if b < len(msg_bits):
                        # Insert 1 space at [even index -> bit 0; odd index -> bit 1]
                        gaps[i * 2 + int(msg_bits[b])] += 1
                        b += 1
                    else:
                        # Bit 0
                        if marker_bit_embedded:
                            gaps[i * 2] += 1
                        # Bit 1
                        else:
                            gaps[i * 2 + 1] += 1
                            marker_bit_embedded = True
                if capacity == n - m:
                    for i in range(capacity * 2, n):
                        gaps[i] += 1
            if m >= n:
                gaps = [gap + (m // n) for gap in gaps]
                gaps[0:(m % n)] = [gap + 1 for gap in gaps[0:(m % n)]]
            stego_line = create_line(words, gaps)
        l += 1
        stego_text += stego_line + ('\n' if l < len(cover_lines) else '')
    
    if not marker_bit_embedded:
        return False
    
    with open(stego_text_file, 'w') as f:
        f.write(stego_text)
    return True
            

**Note:** You need to have the test files (`msg.txt`, `cover.txt`, `correct_stego.txt`, etc.) in the same directory as this notebook.

In [8]:
# OFFICIAL TEST 1: Embedding (5 points)
result = embed('msg.txt', 'cover.txt', 70, 'stego.txt')
assert result == True, "Embedding should succeed"

with open('stego.txt', 'r') as f:
    stego_text = f.read()
with open('correct_stego.txt', 'r') as f:
    correct_stego_text = f.read()
    
assert stego_text == correct_stego_text, "Stego text does not match expected output"
print("Official Test 1 Passed! ✓")

Official Test 1 Passed! ✓


In [9]:
# OFFICIAL TEST 2: Embedding fails when capacity exceeded (1 point)
result = embed('msg2.txt', 'cover.txt', 70, 'stego.txt')
assert result == False, "Embedding should fail due to insufficient capacity"
print("Official Test 2 Passed! ✓")

Official Test 2 Passed! ✓


---
## 7. Task 4: Extract Function (1.5 points)

Implement the extraction function that recovers the hidden message from stego text.

**Requirements:**
1. Read the stego text from `stego_text_file`
2. Parse each line to extract gap sizes
3. Decode bits from gap patterns:
   - Pattern (2, 1) → Bit `0`
   - Pattern (1, 2) → Bit `1`
   - Other patterns → Stop extraction for this line
4. Remove termination pattern (find last `1`, remove it and everything after)
5. Convert bits to text and write to `extr_msg_file`

In [10]:
## This function is becoming quite long; 
## please feel free to decompose it into smaller helper functions as you see fit.

def extract(stego_text_file, extr_msg_file):
    """
    Extract the hidden message from stego text.
    
    Args:
        stego_text_file (str): Path to file containing the stego text.
        extr_msg_file (str): Path to output file for extracted message.
    """
    # Read stego text
    # YOUR CODE HERE
    with open(stego_text_file, 'r') as f:
        stego_text = f.read()
        
    stego_lines = stego_text.splitlines()
    
    extr_msg_bits = ''
    
    l = 0
    
    while l < len(stego_lines):
        if is_aligned_line(l, stego_lines):
            words, gaps = parse_line(stego_lines[l])
            for i in range(0, len(gaps) - 1, 2):
                if gaps[i] == 2 and gaps[i + 1] == 1:
                    extr_msg_bits += '0'
                elif gaps[i] == 1 and gaps[i + 1] == 2:
                    extr_msg_bits += '1'
                else:
                    break
        l += 1
    
    try:
        marker_bit_index = extr_msg_bits.rindex('1') # return -1 if not found
        extr_msg_bits = extr_msg_bits[0:marker_bit_index]
        extr_msg_text = bits_to_text(extr_msg_bits)
        
        with open(extr_msg_file, 'w') as f:
            f.write(extr_msg_text)
    except ValueError:
        print('Extracted message bits is not in the correct format')
    

**Note:** You need to have the test files (`msg.txt`, `cover.txt`, `correct_stego.txt`, etc.) in the same directory as this notebook.

In [11]:
# TEST: Extract from previously embedded message
extract('stego.txt', 'extract_msg.txt')

with open('extract_msg.txt', 'r') as f:
    extracted_msg = f.read()

with open('msg.txt', 'r') as f:
    original_msg = f.read()

assert extracted_msg == original_msg, f"Extracted '{extracted_msg}' != Original '{original_msg}'"

print(f"Original message: '{original_msg}'")
print(f"Extracted message: '{extracted_msg}'")
print("Extraction test passed! ✓")

Original message: 'Be calm'
Extracted message: 'Be calm'
Extraction test passed! ✓


---
## 8. Task 5: Analysis Questions (1 points)

Answer the following questions about the inter-word spacing steganography method.

### Question 1 (0.25 points)

Why is the capacity formula `min(m, n-m)` and not just `m` or `n`?

Explain in terms of the encoding scheme.

**Answer**: For each line:
- `n`: the number of inter-word gaps in a line
- `m`: the number of gaps that receive an extra space
- `n−m`: the number of gaps that do not receive an extra space

In the pair-based encoding scheme, each embedded bit (0 or 1) is represented by a pair of gaps: ***1 extra-space gap*** and ***1 no-extra-space gap***. This means the number of bits we can embed is limited by how many such pairs we can form. We cannot embed more bits than:
- the number of gaps with extra spaces (`m`)
- the number of gaps without extra spaces (`n-m`)

&rarr; Therefore, the maximum embedding capacity is `min(m, n-m)`

### Question 2 (0.25 points)

What are the main weaknesses of this steganography method? List at least 3 scenarios where the hidden data could be lost or detected.

**Answer**:
The main weaknesses of this steganography method:
- Manual editing can break the pattern of data.
- Re-justification / Re-formatting destroys hidden data.
- Whitespace normalization removes all extra spaces.
- Statistical analysis of space distributions can detect hidden data.
- Low capacity with just `min(m, n-m)` per aligned line.

The scenarios where the hidden data could be lost or detected:
- **Manual editing:** If a user edits the text, the extracted message can become meaningless.
- **Re-justification by word processors:** Many word processors will re-justify the text when the font (Monospace fonts use equal-width chars, while other fonts do not), window size, margin change. This destroys the inter-word spacing pattern and lose hidden data.
- **Copy-paste into different applications:** Most applications normalize whitespace (HTML renderer, PDF converter, ...), all the extra spaces will be removed.
- **Statistical detection:** In naturally justified text, extra spaces follow deterministic layout rules. After embedding secret bits, the space distribution deviates from these rules, making the hidden data detectable by statistical analysis.

### Question 3 (0.25 points)

How could you increase the embedding capacity of this method? Propose at least 2 modifications.

**Answer**:
- Use more than one extra space per encoding group: Instead of using exactly one extra space per pair, allow patterns like (3,1), (1,3), (2,2),... and assign multiple bits to each pattern. If an encoding group allows up to $k$ extra spaces distributed across two gaps, there are $(k + 1)$ possible spacing patterns. Therefore, the maximum embedding capacity is approximately $log_2 (k + 1)$ bits per encoding group.
- Use larger gap differences: Instead of encoding one bit per pai,, use gaps (gap1, gap2, gap3) with exactly 2 extra spaces per triple gap to encode 2 bits per triple. As each triple gap has extra 2 bits, we have 6 patterns and can choose 4 among 6 patterns to embed 2 bits.
- Encode every gap independently: Instead of encoding bits using fixed gap pairs, each gap is treated independently with a baseline of 1 space. A fixed number of extra spaces are then distributed among all gaps in the line. Each possible distribution represents a different codeword. With n gaps and m extra spaces, there are $C^{m}_{n}$ possible patterns, allowing approximately $log_2 C^{m}_{n}$ bits to be embedded.

For example, consider the sentence: 'A B C D E'. We have n=4 gaps between the words, so the baseline spacing can be represented as `(1, 1, 1, 1)`. Suppose we want to embed data by adding m=2 extra spaces among these 4 gaps. Each pattern corresponds to selecting two gaps to receive the extra spaces. In total, there are $C^{2}_{4}=6$ possible patterns, so the line can encode approximately $log_2 6 \approx 2.58$ bits of information.

### Question 4 (0.25 points)

Compare this method with zero-width character steganography. What are the advantages and disadvantages of each?

**Answer**:

| | Inter-word spacing | Zero-width characters |
|--------|-------------------|----------------------|
| **Capacity** | Low-medium — limited by `min(m, n−m)` per line | Higher — many zero-width chars can be inserted |
| **Robustness** | Low — very fragile to formatting changes | High — survive copy-paste in most plain-text channels |
| **Invisibility to humans** | Low — spacing can be noticed by careful readers | High — completely invisible to normal readers |
| **Detectability by tools** | Low-medium — simple diff or regex reveals it | Medium — requires a Unicode-aware scanner |
| **Compatibility** | High — works in any plain-text format | Medium — some systems strip or reject zero-width chars |
| **Naturalness** | High — justified text is a natural cover | Low — zero-width chars have no legitimate use in normal prose |
| **Implementation complexity** | Medium - need to justify lines correctly | Low - just insert chars into cover text |

**Inter-word spacing**
- Advantage:
    - Look very natural in justified text
    - No special/invisible Unicode characters are used
- Disadvantage:
    - Extremely fragile to re-formatting, re-justification
    - Can be detected using statistical analysis
    - Low capacity

**Zero-width characters**
- Advantage:
    - Invisible to human eye in almost text viewers and editors
    - Much higher capacity
    - Survive copy-paste in most plain-text channels
- Disadvantage:
    - Can be detected by inspecting Unicode characters in the text
    - Can be removed by tools that normalize/strip invisible Unicode chars